In [1]:
#| default_exp frida

In [1]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate


In [5]:
model_path = 'fred'

In [6]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration 
tokenizer = GPT2Tokenizer.from_pretrained(full_path, eos_token='</s>')
model = T5ForConditionalGeneration.from_pretrained(full_path, torch_dtype=torch.bfloat16) 
device='cuda'
model.to(device);
model.eval();


In [7]:
model

T5ForConditionalGeneration(
  (shared): Embedding(50364, 1536)
  (encoder): T5Stack(
    (embed_tokens): Embedding(50364, 1536)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1536, out_features=1536, bias=False)
              (k): Linear(in_features=1536, out_features=1536, bias=False)
              (v): Linear(in_features=1536, out_features=1536, bias=False)
              (o): Linear(in_features=1536, out_features=1536, bias=False)
              (relative_attention_bias): Embedding(32, 24)
            )
            (layer_norm): FusedRMSNorm(torch.Size([1536]), eps=1e-06, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1536, out_features=4096, bias=False)
              (wi_1): Linear(i

In [8]:
sum(p.numel() for p in model.parameters())

1740354048

In [9]:
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    lm_text = '<LM>' + prompt
    result = model.generate(lm_text, do_sample=True, temperature=.5, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                       max_new_tokens=length, ).generated_text.replace('\n', ' ')
    
    result = process_seq([result])
    return result


In [10]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    return generate(model, tokenizer, seq_length, '<LM>' + prompt, length, num_samples, allow_linebreak)

In [11]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 2.61 s, sys: 188 ms, total: 2.79 s
Wall time: 2.8 s


['аже не может сказать правду в лицо своему начальнику. Но ты все-таки можешь мне помочь… Если тебе нужна моя помощь…» Я подумал: «А ведь это правда!',
 'ел он!» Так что, когда в следующий раз захочешь сказать мне гадости и скажешь их… Ну не буду я на тебя сердиться. Не хочу портить себе карму плохим настроением из-за твоей мерзости!',
 ' не виноват в том… Но ты должен понять и принять то обстоятельство…» А дальше она говорила о себе уже как-будто с другой стороны стекла: «Я люблю тебя за твой ум».',
 ' меняется, как ни крути». Я вспомнил его слова про то же самое и подумал: «Вот оно что!']